## 🧪 Interactive Sandbox
Run the setup cell, then try any query from this chapter with `run("...")`.

In [ ]:
# ── SQL Sandbox — run this cell first ──────────────────────────────
# Creates an in-memory database with the sample tables so you can run the
# queries in this chapter. Use run("SELECT ...") and it returns a DataFrame.
# (SQLite supports SELECT/JOIN/GROUP BY/CTEs/window functions. A few Snowflake-
#  only bits — QUALIFY, DATE_TRUNC, SPLIT_PART — won't run here; those are noted.)
import sqlite3, pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE departments (dept_id INT, dept_name TEXT, location TEXT);
CREATE TABLE employees (emp_id INT, name TEXT, dept_id INT, manager_id INT,
                        salary INT, hire_date TEXT, email TEXT);
CREATE TABLE customers (customer_id INT, name TEXT, city TEXT, signup_date TEXT);
CREATE TABLE products (product_id INT, product_name TEXT, category TEXT, price REAL);
CREATE TABLE orders (order_id INT, customer_id INT, product_id INT, quantity INT,
                     order_date TEXT, amount REAL);
INSERT INTO departments VALUES
 (1,'Engineering','Bangalore'),(2,'Sales','Mumbai'),(3,'Marketing','Delhi'),(4,'HR','Bangalore');
INSERT INTO employees VALUES
 (1,'Asha',1,NULL,180000,'2019-03-01','asha@pjs.com'),
 (2,'Ravi',1,1,120000,'2020-06-15','ravi@pjs.com'),
 (3,'Meera',2,1,95000,'2021-01-20','meera@pjs.com'),
 (4,'Kiran',2,3,70000,'2022-07-11','kiran@pjs.com'),
 (5,'Sana',3,1,88000,'2021-09-05','sana@pjs.com'),
 (6,'Vijay',1,2,110000,'2023-02-28','vijay@pjs.com'),
 (7,'Nina',4,1,60000,'2020-11-30',NULL);
INSERT INTO customers VALUES
 (101,'Rahul','Mumbai','2023-01-10'),(102,'Priya','Delhi','2023-02-14'),
 (103,'Arjun','Bangalore','2023-03-22'),(104,'Divya','Mumbai','2023-05-01');
INSERT INTO products VALUES
 (201,'Laptop','Electronics',55000),(202,'Mouse','Electronics',700),
 (203,'Desk','Furniture',8000),(204,'Chair','Furniture',4500),(205,'Monitor','Electronics',15000);
INSERT INTO orders VALUES
 (1,101,201,1,'2023-06-01',55000),(2,101,202,2,'2023-06-01',1400),
 (3,102,205,2,'2023-06-05',30000),(4,103,203,1,'2023-06-10',8000),
 (5,103,204,4,'2023-06-10',18000),(6,104,201,1,'2023-07-02',55000),
 (7,102,202,1,'2023-07-15',700),(8,101,205,1,'2023-08-01',15000);
""")
def run(sql):
    "Run a SQL query against the sample database and return a DataFrame."
    return pd.read_sql_query(sql, conn)

run("SELECT * FROM employees")  # try it — edit and re-run!

# Module 02 — Filtering, Sorting & Operators

> Rarely do you want *all* rows. This module is about asking for exactly the rows you need, in the order you want.

---

## 2.1 WHERE — Keeping Only the Rows You Want

`WHERE` filters rows by a condition. Only rows where the condition is **TRUE** are returned.

```sql
SELECT name, salary FROM employees
WHERE salary > 100000;
```

`WHERE` runs after `FROM` but before `SELECT` picks columns — it decides which rows survive.

## 2.2 Comparison Operators

| Operator | Meaning |
|----------|---------|
| `=` | equal |
| `<>` or `!=` | not equal |
| `>` `<` `>=` `<=` | greater/less than (or equal) |

```sql
SELECT name FROM employees WHERE dept_id = 1;
SELECT name FROM employees WHERE salary >= 90000;
SELECT name FROM employees WHERE dept_id <> 2;
```

## 2.3 Combining Conditions: AND, OR, NOT

```sql
-- Both must be true
SELECT name FROM employees WHERE dept_id = 1 AND salary > 100000;
-- Either can be true
SELECT name FROM employees WHERE dept_id = 1 OR dept_id = 3;
-- Negate
SELECT name FROM employees WHERE NOT dept_id = 2;
```

> ⚠️ `AND` binds tighter than `OR`. Use parentheses to be explicit:
> `WHERE (dept_id = 1 OR dept_id = 2) AND salary > 90000`.

## 2.4 BETWEEN — Ranges

Inclusive on both ends:

```sql
SELECT name, salary FROM employees
WHERE salary BETWEEN 80000 AND 120000;   -- 80000 and 120000 included
```

## 2.5 IN — Match a List

Cleaner than many `OR`s:

```sql
SELECT name FROM employees WHERE dept_id IN (1, 3);
-- same as: WHERE dept_id = 1 OR dept_id = 3
SELECT name FROM employees WHERE dept_id NOT IN (2, 4);
```

## 2.6 LIKE — Pattern Matching

For partial text matches. Wildcards:
- `%` = any sequence of characters (including none)
- `_` = exactly one character

```sql
SELECT name FROM employees WHERE name LIKE 'A%';    -- starts with A
SELECT name FROM employees WHERE name LIKE '%a';    -- ends with a
SELECT name FROM employees WHERE name LIKE '%in%';  -- contains "in"
SELECT name FROM employees WHERE name LIKE '_a%';   -- 2nd letter is a
```

❄️ Snowflake: `ILIKE` is case-insensitive (`WHERE name ILIKE 'a%'`).

## 2.7 NULL — the Absence of a Value

`NULL` means "unknown / no value" — it is **not** zero or empty string. You **cannot** compare it with `=`:

```sql
-- WRONG: returns nothing, ever
SELECT name FROM employees WHERE email = NULL;
-- RIGHT:
SELECT name FROM employees WHERE email IS NULL;
SELECT name FROM employees WHERE email IS NOT NULL;
```

> ⚠️ Any comparison with NULL yields UNKNOWN (not TRUE), so those rows are excluded. This trips up everyone at first — remember `IS NULL` / `IS NOT NULL`.

## 2.8 ORDER BY — Sorting Results

```sql
SELECT name, salary FROM employees
ORDER BY salary DESC;    -- highest first; ASC (default) = lowest first
```

Sort by multiple columns (tie-breakers):

```sql
SELECT name, dept_id, salary FROM employees
ORDER BY dept_id ASC, salary DESC;   -- by dept, then salary within dept
```

You can order by a column you didn't select, or by position (`ORDER BY 2` = 2nd selected column — handy but less readable).

## 2.9 LIMIT with ORDER BY — Top-N

The classic "top N" pattern: sort, then limit.

```sql
-- Top 3 highest-paid employees
SELECT name, salary FROM employees
ORDER BY salary DESC
LIMIT 3;
```

## 2.10 Order of Operations (mental model)

```
FROM      → get the table
WHERE     → filter rows
SELECT    → pick/compute columns
ORDER BY  → sort
LIMIT     → cap rows
```

This is why you can't use a `SELECT` alias in `WHERE` (WHERE runs first) but often can in `ORDER BY` (which runs after SELECT).

---

## ✅ Key Takeaways
1. `WHERE` filters rows using comparison operators (`=`, `<>`, `>`, …).
2. Combine conditions with `AND`/`OR`/`NOT`; use parentheses to control precedence.
3. `BETWEEN` (ranges), `IN` (lists), `LIKE` (patterns with `%` and `_`).
4. `NULL` is unknown — use `IS NULL` / `IS NOT NULL`, never `= NULL`.
5. `ORDER BY col DESC/ASC` sorts; add columns for tie-breakers.
6. Sort + `LIMIT` = the top-N pattern. Remember the FROM→WHERE→SELECT→ORDER BY→LIMIT order.

## 🏋️ Exercises
1. List employees earning more than 90,000, highest salary first.
2. Find employees in departments 1 or 4 (use `IN`).
3. Find employees whose name contains the letter "a".
4. Find employees with no email (`IS NULL`).
5. Show the 2 cheapest products.
6. List employees hired between '2021-01-01' and '2021-12-31' (use `BETWEEN`).

**Next:** [Module 03 — Aggregations & GROUP BY →](module-03-aggregations.md)

---

*🗄️ SQL Mastery — [PJ's Academy](https://pjsacademy.com)*